## 2.1

In [2]:
import sys

import numpy as np
import pandas as pd

listings = pd.read_csv("data/listings.csv")

In [3]:
listings.shape

(4574, 84)

In [4]:
# show the top five most common property types
listings["property_type"].value_counts().head(5)

property_type
Entire rental unit             2442
Entire home                     569
Private room in rental unit     491
Entire condo                    339
Entire villa                    149
Name: count, dtype: int64

In [5]:
# share of each property type, rounded to three decimals
listings["property_type"].value_counts(normalize=True).round(3)

property_type
Entire rental unit                    0.534
Entire home                           0.124
Private room in rental unit           0.107
Entire condo                          0.074
Entire villa                          0.033
Entire townhouse                      0.016
Room in hotel                         0.014
Private room in condo                 0.011
Private room in home                  0.011
Entire loft                           0.009
Entire serviced apartment             0.009
Private room in bed and breakfast     0.007
Private room in hostel                0.007
Entire guesthouse                     0.006
Tiny home                             0.005
Private room in villa                 0.004
Entire cabin                          0.003
Entire guest suite                    0.003
Room in aparthotel                    0.003
Room in boutique hotel                0.003
Shared room in hotel                  0.002
Boat                                  0.002
Private room in to

## 2.2

In [6]:
listings["neighbourhood_cleansed"].value_counts()

neighbourhood_cleansed
Södermalms                1145
Norrmalms                  469
Hägersten-Liljeholmens     464
Enskede-Årsta-Vantörs      445
Kungsholmens               420
Östermalms                 297
Bromma                     281
Skarpnäcks                 221
Älvsjö                     214
Farsta                     177
Hässelby-Vällingby         148
Rinkeby-Tensta             108
Skärholmens                 96
Spånga-Tensta               89
Name: count, dtype: int64

The three smallest neighbourhoods:
- Rinkeby-Tensta: 108
- Skärholmens: 96
- Spånga-Tensta: 89

## 2.3

In [7]:
# calculate average beds for each room type rounded to two decimals.
listings.groupby("room_type")["beds"].mean().round(2).sort_values()

room_type
Hotel room         1.00
Private room       1.48
Entire home/apt    2.48
Shared room        4.58
Name: beds, dtype: float64

In [8]:
# use agg("count", "mean"]) on the same thing.
(listings.groupby("room_type")["beds"].agg(["count", "mean", "median"])).round(
    2
).sort_values("mean")

,count,mean,median
room_type,,,
Hotel room,3,1.00,1.0
Private room,766,1.48,1.0
Entire home/apt,3680,2.48,2.0
Shared room,24,4.58,4.0


Least trustworthy: Hotel room (1.00), because it is based on only 3 listings, so one odd listing changes the mean a lot. Entire home/apt has the highest count (3680), so single listings have less impact there.

## 2.4

In [10]:
(
    listings.groupby("neighbourhood_cleansed")["number_of_reviews"]
    .agg(["count", "mean", "median"])
    .round(2)
    .sort_values("median")
)

,count,mean,median
neighbourhood_cleansed,,,
Skärholmens,96,21.29,3.0
Farsta,177,18.18,4.0
Hässelby-Vällingby,148,14.94,4.0
Bromma,281,19.16,5.0
Rinkeby-Tensta,108,16.44,5.0
Enskede-Årsta-Vantörs,445,22.55,7.0
Spånga-Tensta,89,17.01,7.0
Hägersten-Liljeholmens,464,22.92,8.0
Skarpnäcks,221,22.55,8.0


Lowest median: Skärholmens (3). Highest median: Södermalms (17).

Södermalms has mean 59.82 but median 17. The median is the middle value once you sort the numbers; the mean adds everything up and divides, so a few listings with very many reviews pull the mean up.

## 2.5

In [11]:
summary = (
    listings.groupby("room_type")
    .agg(
        n_listings=("id", "count"),
        avg_beds=("beds", "mean"),
        max_reviews=("number_of_reviews", "max"),
    )
    .round(2)
)

summary

,n_listings,avg_beds,max_reviews
room_type,,,
Entire home/apt,3760,2.48,1293
Hotel room,3,1.00,79
Private room,787,1.48,867
Shared room,24,4.58,510


## 2.6

In [12]:
(
    pd.crosstab(
        listings["neighbourhood_cleansed"],
        listings["host_is_superhost"],
        normalize="index",
    )
    * 100
).round(1).sort_values("t")

host_is_superhost,f,t
neighbourhood_cleansed,,
Bromma,92.9,7.1
Farsta,91.0,9.0
Kungsholmens,90.2,9.8
Enskede-Årsta-Vantörs,88.3,11.7
Hässelby-Vällingby,87.8,12.2
Hägersten-Liljeholmens,87.5,12.5
Rinkeby-Tensta,87.0,13.0
Skarpnäcks,86.9,13.1
Skärholmens,84.4,15.6


Highest share of superhosts: Södermalms (23.8 %). Lowest: Bromma (7.1 %).

## 2.7

In [13]:
listings["has_reviews"] = listings["number_of_reviews"] > 0

listings["has_reviews"].value_counts()


has_reviews
True     3930
False     644
Name: count, dtype: int64

In [14]:
listings["popularity"] = np.where(
    listings["number_of_reviews"] >= 50, "popular", "other"
)


## 2.8

In [ ]:
# bins start at 1 -> listings with exactly 1 night get no group
wrong = pd.cut(listings["minimum_nights"], bins=[1, 6, 29, 500])
wrong.isna().sum()  # 1331

np.int64(1331)

In [20]:
counts = listings["stay"].value_counts().sort_index()
missing = listings["stay"].isna().sum()

print("Listings per stay group:")
print(counts.to_string())
print(f"\nNo group: {missing}")


Listings per stay group:
stay
1 night        1327
2-6 nights     2661
7-29 nights     430
30+ nights      152

No group: 4
